# Natural-Language MCP Authoring With OpenAI

This generated notebook is the recipe companion for
`examples/mcp_natural_language_demo.py`.

Use the public `wellplot.agent` API to drive local `wellplot-mcp` authoring from natural-language instructions and recreate a production example variant without embedding provider or MCP glue in the notebook.

What you will practice in this walkthrough:

- Create one public `AuthoringSession` backed by local stdio MCP and the OpenAI adapter.
- Run one natural-language authoring request against a packaged production example.
- Inspect the tool trace, structural change summary, and rendered previews from the returned `AuthoringResult`.
- Persist the in-memory preview PNGs next to the generated draft logfile through the result helper.

Prerequisites:

- `pip install "wellplot[agent,las,notebook]"`
- run the notebook from a checkout of this repository so the `examples/` files and sample data are available

Runtime model:

- import `wellplot` from the active installed environment
- use the repository checkout for the example files, helper modules, and sample data


In [ ]:
import sys
from pathlib import Path

try:
    import wellplot
except ImportError as exc:
    raise RuntimeError(
        "Install the published 'wellplot' package in the active "
        "environment before running this notebook."
    ) from exc

# Walk upward from the current working directory until we find the
# repository checkout that holds the example sources and sample data.
cwd = Path.cwd().resolve()
REPO_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "examples").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "Run this notebook from a checkout of the wellplot repository "
        "so the example files and sample data are available."
    )

EXAMPLES_DIR = REPO_ROOT / "examples"
WORKSPACE_DIR = REPO_ROOT / "workspace"
WORKSPACE_RENDERS = WORKSPACE_DIR / "renders"
WORKSPACE_RENDERS.mkdir(parents=True, exist_ok=True)

examples_path = str(EXAMPLES_DIR)
if examples_path not in sys.path:
    sys.path.insert(0, examples_path)

print("wellplot version:", wellplot.__version__)
print("Examples root:", EXAMPLES_DIR)
print("Render output:", WORKSPACE_RENDERS)

In [ ]:
# Display the source directly in the notebook so the recipe is easy to
# read and copy from without opening another file.
from IPython.display import Code, display

source_path = EXAMPLES_DIR / "mcp_natural_language_demo.py"
display(Code(source_path.read_text(), language="python"))

In [ ]:
# Import the public authoring API from the installed package.
import os
from pathlib import Path
from wellplot.agent import AuthoringSession

DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
DEFAULT_EXAMPLE_ID = "forge16b_porosity_example"
DEFAULT_OUTPUT_LOGFILE = Path("workspace/mcp_demo/openai_forge16b_recreated.log.yaml")

In [ ]:
# No notebook-local credential prompt is needed here.
# `wellplot.agent` will load OpenAI credentials from the same
# ignored local sources as the rest of the package:
# `OPENAI_API_KEY`, `.env.local`, `.env`, `OPENAI_API_KEY.txt`,
# or `openai_api_key.txt` under the configured server root.


In [ ]:
# Edit these values before rerunning the notebook on a different target.
GOAL = (
    "Recreate the forge16b porosity example as a simplified interpretation "
    "packet. Keep one GR/SP track, one depth track, one resistivity track, "
    "and one porosity overlay track with RHOB and NPHI. Shorten the remarks "
    "to one concise block and simplify the heading."
)
EXAMPLE_ID = DEFAULT_EXAMPLE_ID
MODEL = DEFAULT_MODEL
OUTPUT_LOGFILE = DEFAULT_OUTPUT_LOGFILE

print("Model:", MODEL)
print("Seed example:", EXAMPLE_ID)
print("Draft output:", OUTPUT_LOGFILE)
print("Server root:", REPO_ROOT)

session = AuthoringSession.from_local_mcp(
    provider="openai",
    model=MODEL,
    server_root=REPO_ROOT,
)

In [ ]:
# Run the natural-language authoring loop through the public API.
from IPython.display import Code, Image, Markdown, display

authoring = await session.run(
    goal=GOAL,
    example_id=EXAMPLE_ID,
    output_logfile=OUTPUT_LOGFILE,
)

preview_paths = authoring.write_preview_artifacts()

print("Provider:", authoring.provider)
print("Model:", authoring.model)
print("Token source:", authoring.credential_source)
print("Draft logfile:", authoring.draft_logfile)
print("Preview files:")
for name, path in preview_paths.items():
    print(f" - {name}: {path.relative_to(REPO_ROOT)}")

print("\nValidation:", authoring.validation)
print("\nModel summary:\n")
display(Markdown(authoring.final_text or "_No final text returned._"))

print("\nTool trace:")
for item in authoring.tool_trace:
    print(f" - round {item.round}: {item.name}({item.arguments})")

print("\nChange summary:")
for line in authoring.change_summary.get("summary_lines", []):
    print(" -", line)

print("\nFirst section ids:", authoring.inspect_summary["section_ids"])
display(Image(data=authoring.report_preview_png))
display(Image(data=authoring.section_preview_png))

preview_yaml = "\n".join(authoring.draft_text.splitlines()[:120])
display(Code(preview_yaml, language="yaml"))

## Adapt Natural-Language MCP Authoring With OpenAI Safely

- Credentials are loaded through `wellplot.agent`, so the notebook works with any supported local source: `OPENAI_API_KEY`, `.env.local`, `.env`, `OPENAI_API_KEY.txt`, or `openai_api_key.txt` under the repository or job root.
- Use `.env.local` or `OPENAI_API_KEY.txt` when you want one persistent local secret that stays out of version control without editing notebook cells.
- Start from `forge16b_porosity_example` or another LAS-backed production packet first so the natural-language loop stays fast and reproducible.
- Treat the OpenAI model as the planner and `wellplot-mcp` as the deterministic executor; if the result is close but not right, rerun with a tighter goal or follow up with `revise_plot_from_feedback(...)` through the MCP prompt layer.
